# Part 5 — Market Basket Analysis

### Goal
Discover products frequently purchased together. Business question:

"What products should we bundle or recommend together?"

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules


### 1. Load and prepare data 

In [2]:
df = pd.read_csv('../data/online_retail_cleaned.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,transactiontype
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,Purchase
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,Purchase
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase


In [3]:
print(df.shape)

(401604, 10)


In [4]:
#drop missing values:
basket_data = df.dropna(subset=['InvoiceNo', 'Description']).copy()
basket_data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,transactiontype
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,Purchase
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,Purchase
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase


In [5]:
# Remove canceled transactions and returned items
basket_data = basket_data[
    (~basket_data['InvoiceNo'].astype(str).str.startswith('C')) &
    (basket_data['Quantity'] > 0)
]

### 2. Create Basket Dataset

Creating  a basket dataset:    
Rows: Transactions (invoiceID)  
Columns: Products (description)  
Values: Purchased or not purchased based on 'Quality' columns:  
quantity >0 : purchased (1)  
quantity = 0 : not purchased (0)   

After creating the basket, we apply  Association Rules / Apriori to discover products chosen together.

In [6]:
df['Description'].unique()

<StringArray>
[ 'WHITE HANGING HEART T-LIGHT HOLDER',                 'WHITE METAL LANTERN',
      'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE',
      'RED WOOLLY HOTTIE WHITE HEART.',        'SET 7 BABUSHKA NESTING BOXES',
   'GLASS STAR FROSTED T-LIGHT HOLDER',              'HAND WARMER UNION JACK',
           'HAND WARMER RED POLKA DOT',       'ASSORTED COLOUR BIRD ORNAMENT',
 ...
     'SILVER AND BLACK ORBIT NECKLACE',           'LETTER "F" BLING KEY RING',
           'LETTER "O" BLING KEY RING',           'LETTER "T" BLING KEY RING',
           'LETTER "U" BLING KEY RING',           'LETTER "W" BLING KEY RING',
           'LETTER "Z" BLING KEY RING',      'PINK CRYSTAL SKULL PHONE CHARM',
  'CREAM HANGING HEART T-LIGHT HOLDER',         'PAPER CRAFT , LITTLE BIRDIE']
Length: 3896, dtype: str

In [7]:
basket = basket_data.pivot_table(
    index='InvoiceNo',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)

In [8]:
basket = basket.gt(0).astype(int)

In [9]:
basket.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
basket.tail()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581585,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
581586,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581587,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
basket.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
#to check  unique values 
basket.stack().unique()
basket.dtypes

Description
 4 PURPLE FLOCK DINNER CANDLES      int64
 50'S CHRISTMAS GIFT BAG LARGE      int64
 DOLLY GIRL BEAKER                  int64
 I LOVE LONDON MINI BACKPACK        int64
 I LOVE LONDON MINI RUCKSACK        int64
                                    ...  
ZINC T-LIGHT HOLDER STARS SMALL     int64
ZINC TOP  2 DOOR WOODEN SHELF       int64
ZINC WILLIE WINKIE  CANDLE STICK    int64
ZINC WIRE KITCHEN ORGANISER         int64
ZINC WIRE SWEETHEART LETTER TRAY    int64
Length: 3877, dtype: object

In [13]:
basket = basket.astype(bool)
basket.dtypes

Description
 4 PURPLE FLOCK DINNER CANDLES      bool
 50'S CHRISTMAS GIFT BAG LARGE      bool
 DOLLY GIRL BEAKER                  bool
 I LOVE LONDON MINI BACKPACK        bool
 I LOVE LONDON MINI RUCKSACK        bool
                                    ... 
ZINC T-LIGHT HOLDER STARS SMALL     bool
ZINC TOP  2 DOOR WOODEN SHELF       bool
ZINC WILLIE WINKIE  CANDLE STICK    bool
ZINC WIRE KITCHEN ORGANISER         bool
ZINC WIRE SWEETHEART LETTER TRAY    bool
Length: 3877, dtype: object

In [14]:
#Apply apriori association with bool variables
frequent_itemsets = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

In [15]:
#display result :frequent items
frequent_itemsets.sort_values(
    by='support',
    ascending=False
).head(10)

,support,itemsets
587,0.106334,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
414,0.091929,frozenset({REGENCY CAKESTAND 3 TIER})
247,0.086319,frozenset({JUMBO BAG RED RETROSPOT})
338,0.074450,frozenset({PARTY BUNTING})
38,0.074180,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
282,0.069486,frozenset({LUNCH BAG RED RETROSPOT})
468,0.061826,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN })
374,0.059290,frozenset({POSTAGE})
274,0.056754,frozenset({LUNCH BAG BLACK SKULL.})
323,0.055514,frozenset({PACK OF 72 RETROSPOT CAKE CASES})


### Deliverable: 


Generate Association Rules:
Apriori finds frequent product combinations, but Association Rules help us understand relationships between products.

In [16]:
#analyse support, confidence, lift
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.3
)

In [17]:
#display results 
rules = rules[
    ['antecedents', 'consequents',
     'support', 'confidence', 'lift']
]

rules.head()

,antecedents,consequents,support,confidence,lift
0,frozenset({60 CAKE CASES DOLLY GIRL DESIGN}),frozenset({PACK OF 72 RETROSPOT CAKE CASES}),0.010196,0.533898,9.617433
1,frozenset({60 CAKE CASES VINTAGE CHRISTMAS}),frozenset({SET OF 20 VINTAGE CHRISTMAS NAPKINS}),0.010088,0.401288,15.211178
2,frozenset({SET OF 20 VINTAGE CHRISTMAS NAPKINS}),frozenset({60 CAKE CASES VINTAGE CHRISTMAS}),0.010088,0.382413,15.211178
3,frozenset({72 SWEETHEART FAIRY CAKE CASES}),frozenset({60 TEATIME FAIRY CAKE CASES}),0.011977,0.443114,12.501609
4,frozenset({60 TEATIME FAIRY CAKE CASES}),frozenset({72 SWEETHEART FAIRY CAKE CASES}),0.011977,0.337900,12.501609


#### Find the strongest association :
support represents how many frequent these 2 products appear together in all transactions.  
confidence represents persentage of buying product A the product B.  shoud be >50%  
lift : how strongly A and B are associated accross all of transactions of A and B separeted:  
lift > 1 :positive association, lift =1 : no association, lift <1 : negative association.

In [18]:
#analyse ad filter results, sorted by lift 
strong_rules = rules[
    (rules['confidence'] >= 0.5) &
    (rules['lift'] > 1)
].sort_values(
    by='lift',
    ascending=False
)
# strong_rules = strong_rules.reset_index()
strong_rules.head(10)

,antecedents,consequents,support,confidence,lift
364,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE GREEN }),0.010898,0.901786,61.909259
365,frozenset({REGENCY TEA PLATE GREEN }),frozenset({REGENCY TEA PLATE PINK}),0.010898,0.748148,61.909259
585,"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),0.010035,0.732283,53.863517
586,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",0.010035,0.738095,53.863517
358,frozenset({REGENCY MILK JUG PINK }),frozenset({REGENCY SUGAR BOWL GREEN}),0.011114,0.757353,52.381694
359,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK }),0.011114,0.768657,52.381694
588,frozenset({POPPY'S PLAYHOUSE BEDROOM }),"frozenset({POPPY'S PLAYHOUSE LIVINGROOM , POPP...",0.010035,0.588608,50.746188
583,"frozenset({POPPY'S PLAYHOUSE LIVINGROOM , POPP...",frozenset({POPPY'S PLAYHOUSE BEDROOM }),0.010035,0.865116,50.746188
369,frozenset({REGENCY TEA PLATE ROSES }),frozenset({REGENCY TEA PLATE PINK}),0.010628,0.600610,49.700457
368,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE ROSES }),0.010628,0.879464,49.700457


In [22]:
strong_rules[['antecedents', 'consequents']].head(10)

,antecedents,consequents
364,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE GREEN })
365,frozenset({REGENCY TEA PLATE GREEN }),frozenset({REGENCY TEA PLATE PINK})
585,"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",frozenset({POPPY'S PLAYHOUSE LIVINGROOM })
586,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ..."
358,frozenset({REGENCY MILK JUG PINK }),frozenset({REGENCY SUGAR BOWL GREEN})
359,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK })
588,frozenset({POPPY'S PLAYHOUSE BEDROOM }),"frozenset({POPPY'S PLAYHOUSE LIVINGROOM , POPP..."
583,"frozenset({POPPY'S PLAYHOUSE LIVINGROOM , POPP...",frozenset({POPPY'S PLAYHOUSE BEDROOM })
369,frozenset({REGENCY TEA PLATE ROSES }),frozenset({REGENCY TEA PLATE PINK})
368,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE ROSES })


After Sorting the dataset by 'lift' descending, we find the strongest rule :  

rule 1 :  
frozenset({REGENCY TEA PLATE GREEN }) -->	frozenset({REGENCY TEA PLATE PINK})   
that appears in ~11% in all transations, with confidence : ~75% > 50% and lift ~ 62 >>1    
SO, these mertices indicates strong positive association.   

rule 2 :
frozenset({REGENCY TEA PLATE PINK})	--> frozenset({REGENCY TEA PLATE GREEN })  
  
that appears in ~11% in all transations, with confidence : ~90% > 50% and lift ~ 62 >>1    
SO, these mertices indicates strong positive association.  

the table shows top 10 of strongest rules. SO, The company should create a bundle offers by grouping products having strong associations together. 

In [19]:
strong_rules.to_csv('strongest_rules.csv', index = False)

In [20]:
import pickle
strong_rules.to_pickle('strong_rules.pkl')